# pipeline_driver — registry-driven medallion pipeline (Snowflake)

Runs one system's contracts through the requested layers. Edit the contracts, not this notebook.
> generated by: LakeLogic Medallion Builder
> revision: chv1:a5264ea0ab53dafe06f0059818462cccb7f8c7f4df07d6b7e974488159c5b1df
> governance seal: sha256:4eb207b04b65320c4d32ffba41eac7bae8a56f63d105b18e6df1f2a3b308db9c
> source: estate snapshot snap-429b586fd4c04d3d

In [ ]:
# Parameters. A task passes them as one key=value string:
#   EXECUTE NOTEBOOK <DB>.PIPELINE.<NB>('key=value key=value')
# For an interactive run, edit the defaults below.
import sys
# ALL of sys.argv: Snowflake fills it from index 0 (no notebook name in front), so
# skipping index 0 drops the first argument. key=value parsing does not care where each sits.
_kv = dict(t.split('=', 1) for t in ' '.join(sys.argv).split() if '=' in t)
DOMAIN = _kv.get('domain', '')  # the domain whose _system.yaml to load
SYSTEM = _kv.get('system', '')  # the system within that domain
DB = _kv.get('db', '')  # the Snowflake database (the task passes it)
ENVIRONMENT = _kv.get('environment', 'dev')  # a key of _system.yaml environments
LAYER = _kv.get('layer', 'all')  # bronze | silver | gold | all
print({'domain': DOMAIN, 'system': SYSTEM, 'db': DB, 'environment': ENVIRONMENT, 'layer': LAYER})

In [ ]:
# lakelogic==1.79.6 from PyPI (deployment target); if this account's notebooks cannot reach PyPI,
# from wheels staged in <DB>.PIPELINE.WHEELS (see tasks/admin/_setup.sql).
import os, subprocess, sys
def _pip(*a):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *a]).returncode
if _pip('lakelogic==1.79.6', 'polars', 'pandas', 'pyarrow') != 0:
    from snowflake.snowpark.context import get_active_session
    _s = get_active_session(); os.makedirs('/tmp/wl', exist_ok=True)
    _s.file.get(f'@{DB}.PIPELINE.WHEELS', '/tmp/wl')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index',
                    '--find-links', '/tmp/wl', 'lakelogic==1.79.6'], check=True)

In [ ]:
# The registry paths are relative to the repository root; a task's CWD is not guaranteed to be it.
import os, sys
d = os.getcwd()
while d != os.path.dirname(d) and not os.path.isdir(os.path.join(d, '_contracts')):
    d = os.path.dirname(d)
REPO = d; sys.path.insert(0, REPO); os.chdir(REPO)
SYS = f'_contracts/{DOMAIN}/{SYSTEM}/_system.yaml'
os.environ['SNOWFLAKE_DATABASE'] = DB
from snowflake.snowpark.context import get_active_session
session = get_active_session()
conn = session.connection
session.sql(f'USE DATABASE {DB}').collect()
session.sql(f'USE SCHEMA {DB}.PIPELINE').collect()
from lakelogic.engines.snowflake import SnowflakeAdapter
SnowflakeAdapter.set_shared_connection(conn)

In [ ]:
from runtime.medallion import run_layer
import pandas as pd
layers = ['bronze', 'silver', 'gold'] if LAYER == 'all' else [LAYER]
rows = [(L,) + tuple(r) for L in layers for r in run_layer(conn, DB, SYS, L, environment=ENVIRONMENT)]
df = pd.DataFrame(rows, columns=['layer', 'entity', 'status', 'good', 'quarantined'])
failed = df[~df['status'].eq('ok')]
if len(failed):
    raise RuntimeError(f'{len(failed)} contract(s) failed — ' + ' | '.join(
        f"{e} [{L}]: {s}" for e, L, s in zip(failed['entity'], failed['layer'], failed['status'])))
df